# Version1 / ZU7CG / PYNQ 4.0：Llama3.2-3B 推理

本部署目录是独立的 llama32_3b_ctx8192_250mhz，核心目标时钟 187.5 MHz。当前仅有源码与配置，没有可下载 bit；先完成同名 Vivado 工程的综合、实现、最终时序验收和配套 bit/HWH/manifest 导出，再运行本 Notebook。PS 高端 2 GiB 必须在启动设备树中专用 no-map 预留；请用 assert_ps_reserved() 核验实际运行配置。两份模型各 1,427,628,032 字节，需要另外提供。主机测试不代表板端推理验收。


硬件导出入口（在 Version1 根目录运行）：`python3 tools/pynq/export_llama32_overlay.py --profile pynq --impl-run impl_2 --project-dir vivado_prj/llama32_3b_ctx8192_250mhz`。共享 export_overlay.py 对应 7B。

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "config/pynq.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "config/pynq.json").is_file():
    raise RuntimeError("请从 llama32_3b_ctx8192_250mhz 或其 notebooks 目录打开此 Notebook")
sys.path.insert(0, str(ROOT))
from llama32_pynq import BoardConfig
from llama32_pynq.board import PynqSession, inspect_files
config = BoardConfig.load(ROOT / "config/pynq.json")
print(ROOT)


## 1. 文件及 prompt 预检（无需板卡）

缺少新 bit / HWH / manifest 或任意模型时先停止，不会自动用原裸机 bit 推理。

In [ ]:
assets = inspect_files(config)
print(json.dumps(assets, ensure_ascii=False, indent=2))
assert assets["inference_assets_ready"], "请先完成硬件与模型准备"
prompt = "Explain what an FPGA is."
max_new_tokens = 32
session = PynqSession(config)
print("prompt tokens:", session.prepare_prompt(prompt, max_new_tokens))


## 2. 验证启动预留并连接（板端）

此步骤下载 Overlay。请确保无其他程序使用 PL。系统需提供 FPGA manager 和 /dev/mem 访问权限。DDR 小范围检查将保存、写入、读回并恢复 PS / PL 专用区域末页的 64 字节；首次访问需要 cc 或 gcc。

In [ ]:
from llama32_pynq.memory import assert_ps_reserved
print(assert_ps_reserved())
print(json.dumps(session.connect(), ensure_ascii=False, indent=2))
print(json.dumps(session.ddr_smoke_test(), ensure_ascii=False, indent=2))


## 3. 加载并完整读回校验模型

模型内容保持原 Vitis PS+PL 打包布局，从两个区域起点连续加载。无需额外填充或再次交织。可提供已知的 expected_hashes，格式如 {"ps": "64位SHA256", "pl": "64位SHA256"}。

In [ ]:
last_progress = {}
def progress(bank, written, total):
    if written == total or written - last_progress.get(bank, 0) >= 256 * 1024 * 1024:
        print(f"{bank.upper()}: {written / 2**20:.0f}/{total / 2**20:.0f} MiB")
        last_progress[bank] = written

load_report = session.load_models(progress=progress)
print(json.dumps(load_report, indent=2))


## 4. 单轮生成

成功加载模型后可重复执行此单元并修改 prompt。每次从空闲状态重新开始单轮对话，不保留多轮上下文。首 token 延迟包含 prefill；decode 速率按输出 token 到达间隔计算。

In [ ]:
result = session.generate(prompt, max_new_tokens=max_new_tokens)
print(result.text)
print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))


超时或控制回调异常后会话将锁定，不会自动复位或覆盖模型。原 RTL 没有排空未完成 AXI 的中止协议；确认板卡恢复后重新连接并加载。重新配置 PL 后旧会话失效；重新运行连接单元时需新建 PynqSession。若 probe 无响应或 Jupyter 断连，停止重复访问并使用独立串口检查系统。不要下载旧工程的 bit 或将其 HWH 与本工程混用。
